# Analysis of experiment #1: DPCP heuristics

## Read data

In [1]:
# Read csv

import pandas as pd

df = pd.read_csv("stats3.csv")

# Some renames to improve readability
df["solver"] = df["solver"].str.replace("v2", "deg")
df["solver"] = df["solver"].str.replace("v3", "edg")

# For the unsolved instances,
# --> write NaN in ub, bestTime, bestIter
df.loc[df.state != "FEASIBLE", "value"] = float("nan")
df.loc[df.state != "FEASIBLE", "bestTime"] = float("nan")
df.loc[df.state != "FEASIBLE", "bestIter"] = float("nan")

# Rename some columns
df = df.rename(columns={'state': 'solved'})

# Add column with edge probability
df["p"] = df.instance.apply(lambda s: float(s.split("_")[2][1:]))

df

,instance,solver,run,nvertices,nedges,n,m,solved,value,totalTime,totalIters,bestTime,bestIter,p
0,r_n170_p0.75_nA17_nB34_i1,heur_greedy2s_v4,0,170,10781,17,34,FEASIBLE,6.0,0.003840,1,0.003840,0.0,0.75
1,r_n170_p0.75_nA17_nB17_i1,heur_greedy2s_v4,0,170,10787,17,17,FEASIBLE,7.0,0.003637,1,0.003636,0.0,0.75
2,r_n130_p0.25_nA26_nB26_i1,heur_greedy2s_v4,0,130,2071,26,26,FEASIBLE,4.0,0.000920,1,0.000919,0.0,0.25
3,r_n150_p0.5_nA15_nB30_i2,heur_greedy2s_v4,0,150,5529,15,30,FEASIBLE,4.0,0.001334,1,0.001334,0.0,0.50
4,r_n150_p0.5_nA15_nB15_i4,heur_greedy2s_v4,0,150,5682,15,15,FEASIBLE,4.0,0.001288,1,0.001288,0.0,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,r_n130_p0.25_nA26_nB26_i2,heur_greedy1s,0,130,2100,26,26,FEASIBLE,3.0,0.002944,1,0.002944,0.0,0.25
346,r_n170_p0.75_nA34_nB17_i1,heur_greedy1s,0,170,10786,34,17,UNKNOWN,NaN,0.009998,1,NaN,NaN,0.75
347,r_n150_p0.5_nA30_nB30_i0,heur_greedy1s,0,150,5548,30,30,FEASIBLE,7.0,0.006714,1,0.006714,0.0,0.50
348,r_n150_p0.5_nA30_nB15_i1,heur_greedy1s,0,150,5541,30,15,UNKNOWN,NaN,0.005603,1,NaN,NaN,0.50


## Best 

In [2]:
# Pivot dataframe
df2 = df.pivot(index="instance", columns="solver", values=["value","totalTime"])
n_heurs = int(len(df2.columns)/2)

In [3]:
import numpy

# Comparison function
def get_winner_heur(row):
    best_heur = None
    min_value = float("inf")
    min_time = float("inf")
    for i in range(n_heurs):
        h = df2.columns[i][1] # get heuristic's name
        j = ("totalTime", h) # get column's name of the bestTime for h
        if (row.iloc[i] < min_value) or (row.iloc[i] == min_value and row[j] < min_time):
            best_heur = h
            min_value = row.iloc[i]
            min_time = row[j]
    return best_heur

def get_winner_value(row):
    values = [row[i] for i in range(n_heurs)]
    return numpy.nanmin(values)

In [4]:
df2["winnerValue"] = df2.apply(get_winner_value, axis=1)
df2["winner"] = df2.apply(get_winner_heur, axis=1)

In [5]:
df["winner"] = df.apply(lambda row: row.solver == df2.loc[row.instance].winner, axis=1)
df["tie"] = df.apply(lambda row: (row.value == df2.loc[row.instance].winnerValue), axis=1)

## Summary

In [6]:
df3 = df.groupby(["solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "winner": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "tie": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "value": [("sum", "sum")],
        "totalTime": [("avg", "mean")],
    }
).reset_index()
df3

,solver,solved,winner,tie,value,totalTime
,,count,%,%,sum,avg
0,heur_greedy1s,36,14.0,20.0,197.0,0.007421
1,heur_greedy2s_deg,46,0.0,0.0,303.0,0.002062
2,heur_greedy2s_edg,46,6.0,6.0,306.0,0.001938
3,heur_greedy2s_v4,46,0.0,0.0,326.0,0.002236
4,heur_semigreedy2s_deg,50,18.0,86.0,269.0,76.239318
5,heur_semigreedy2s_edg,50,54.0,82.0,272.0,59.730520
6,heur_semigreedy2s_v4,50,8.0,72.0,276.0,71.273628


In [7]:
df3 = df.groupby(["p","solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "winner": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "tie": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "value": [("sum", "sum")],
        "totalTime": [("avg", "mean")],
    }
).reset_index()
df3

,p,solver,solved,winner,tie,value,totalTime
,,,count,%,%,sum,avg
0,0.25,heur_greedy1s,7,20.0,30.0,26.0,0.002711
1,0.25,heur_greedy2s_deg,10,0.0,0.0,45.0,0.000903
2,0.25,heur_greedy2s_edg,10,10.0,10.0,43.0,0.001328
3,0.25,heur_greedy2s_v4,10,0.0,0.0,47.0,0.000852
4,0.25,heur_semigreedy2s_deg,10,0.0,90.0,31.0,29.996390
5,0.25,heur_semigreedy2s_edg,10,70.0,100.0,30.0,22.389570
6,0.25,heur_semigreedy2s_v4,10,0.0,100.0,30.0,28.180630
7,0.50,heur_greedy1s,15,10.0,15.0,68.0,0.005267
8,0.50,heur_greedy2s_deg,20,0.0,0.0,120.0,0.001881


## Semigreedy
### Compute % of improvement in value

In [ ]:
# Filter data whose solver is the semigreedy heuristic 
df_sg = df[df.solver.str.contains("semigreedy")]

# Add a column with the name of the solver but when 10 repetitions are performed
def get_solver10(name):
    tokens = name.split("_")
    tokens[3] = "r10"
    return "_".join(tokens)
df_sg["solver10"] = df_sg.solver.apply(lambda name: get_solver10(name))
df_sg

In [ ]:
# Filter data whose solver is the semigreedy heuristic with 10 repetitions
df_sg_r10 = df_sg[df_sg.solver.isin(["heur_semigreedy2s_deg_r10", "heur_semigreedy2s_edg_r10"])]
df_sg_r10

In [ ]:
# Add a column with the value found by the same solver but when 10 repetitions are performed
df_sg["baseValue"] = df_sg.apply(lambda row: float(df_sg_r10[(df_sg_r10.instance == row.instance) 
                                                       & (df_sg_r10.solver == row.solver10)].value), axis = 1)
# Add a column with the improvement (%) in the value with respect to the value found 
# by the same solver but when 10 repetitions are performed
df_sg["improvement"] = df_sg.apply(lambda row: 100 - (row.value * 100 / row.baseValue), axis = 1)
df_sg

In [ ]:
## TODO: ¿que hacemos con los nan (value.isna())?

In [ ]:
# Resumen de los resultados
df_sg2 = df_sg.groupby(["nvertices","solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "value": [("sum","sum"), ("avg", "mean")],
        "improvement": [("avg (%)", "mean")],
        "totalTime": [("avg", "mean")],
        "totalIters": [("avg", "mean"),("max", "max")],
        "bestIter": [("avg","mean"),("max", "max")]
    }
).reset_index()
df_sg2

In [ ]:
# Apartir de estos resultados, filtramos los semigreedy que no hagan 10000 iteraciones
rs = [10,100,1000,100000]
sg_heurs = []
for v in ["deg", "edg"]:
    for r in rs:
        sg_heurs.append("heur_semigreedy2s_"+v+"_r"+str(r))
df = df[~df.solver.isin(sg_heurs)]
df